# Stage 2: Per-Detection Concatenated Embeddings (Yucatan)

Reads per-detection features from `batdetect2_detections.csv` (created by
`embed_stage1_batdetect.ipynb`) and builds a Hoplite embedding database with
one **concatenated** embedding per 0.5s window.

- **Windows with detections**: sort by confidence (descending), take top MAX_DET,
  L2-normalise each 32-dim feature individually, concatenate, zero-pad to
  (MAX_DET x 32)-dim, then L2-normalise the full vector
- **Windows with NO detections**: sentinel vector `np.full(MAX_DET*32, -1.0)`


## 1. Environment Setup

In [ ]:
# Optional: Disable TensorFlow GPU and XLA for CPU-only environments
import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

In [ ]:
# Core imports
import os
import shutil
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
from ml_collections import config_dict
from tqdm.auto import tqdm

# Hoplite DB
from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.db import interface as db_interface
from perch_hoplite.agile import source_info

print("Imports successful! (No BatDetect2 needed — reading from CSV)")

## 2. Configuration

In [ ]:

# USER CONFIGURATION


# Stage 1 CSV files 
DETECTIONS_CSV = "C:/Users/Administrator/hello/Yucatan/batdetect2_detections.csv"
FILE_INDEX_CSV = "C:/Users/Administrator/hello/Yucatan/batdetect2_file_index.csv"

# Path for the Hoplite embedding database
DB_PATH = "C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings_BatDetect2_concat"

# Audio source info (for hoplite metadata -- must match Stage 1 config)
DATASET_NAME = "yucatan_bat_train"
DATASET_BASE_PATH = "C:/Users/Administrator/hello/Yucatan/acoustic_data/data/yucatan/audio"
FILE_GLOB = "*.wav"


# CONCATENATION CONFIGURATION


# Base feature dimension from BatDetect2 CNN
BASE_DIM = 32

# Maximum detections per window 
MAX_DET = 8

# Final embedding dimension = MAX_DET * BASE_DIM
EMBEDDING_DIM = MAX_DET * BASE_DIM  # = 256

# Detection sorting
SORT_BY = 'class_prob'
SORT_ASCENDING = False

# Window size and hop
WINDOW_SIZE_S = 0.5
HOP_SIZE_S = 0.5  # non-overlapping

# Normalisation
NORMALIZE_EMBEDDINGS = True

# Sentinel for empty windows (no detections)
SENTINEL_VALUE = -1.0
SENTINEL_VECTOR = np.full(EMBEDDING_DIM, SENTINEL_VALUE, dtype=np.float32)


# DATABASE OPTIONS


DELETE_EXISTING_DB = True
USEARCH_DTYPE = 'float16'
USEARCH_METRIC = 'IP'


# STAGE 1 PARAMETERS 
TIME_EXPANSION = 1.0
DETECTION_THRESHOLD = 0.3
TARGET_SAMP_RATE = 256000
CHUNK_SIZE = 3.0

print("Configuration loaded.")
print(f"  Detections CSV: {DETECTIONS_CSV}")
print(f"  File index CSV: {FILE_INDEX_CSV}")
print(f"  Database path:  {DB_PATH}")
print(f"  Window size:    {WINDOW_SIZE_S}s (hop={HOP_SIZE_S}s)")
print(f"  MAX_DET:        {MAX_DET} (concat dim = {EMBEDDING_DIM})")
print(f"  Sort by:        {SORT_BY} ({'ascending' if SORT_ASCENDING else 'descending'})")
print(f"  Embedding dim:  {EMBEDDING_DIM}")
print(f"  Sentinel:       [{SENTINEL_VALUE}] * {EMBEDDING_DIM}")
print(f"  Normalise:      {NORMALIZE_EMBEDDINGS}")

## 3. Embedding Function

Groups detections by window, **concatenates** (not averages) top-K features,
zero-pads, and emits sentinel for empty windows.
No BatDetect2 inference needed -- just pandas grouping and numpy operations.

In [ ]:
def embeddings_from_csv_concat(
    file_df: pd.DataFrame,
    duration_s: float,
    feat_cols: List[str],
    window_size_s: float = 0.5,
    hop_size_s: float = 0.5,
    max_det: int = 8,
    sort_by: str = 'class_prob',
    sort_ascending: bool = False,
    normalize: bool = True,
    sentinel: Optional[np.ndarray] = None,
) -> List[Tuple[float, np.ndarray, int, bool]]:
    '''Convert CSV rows for one file into per-window concatenated embeddings.

    For each 0.5s window:
    1. Gather detections, sort by confidence (descending)
    2. Take top max_det detections
    3. L2-normalise each 32-dim feature individually
    4. Concatenate into (max_det * 32)-dim vector, zero-pad if K < max_det
    5. L2-normalise the full concatenated vector

    Returns:
        List of (window_offset_s, embedding, n_detections, was_truncated) tuples.
    '''
    base_dim = len(feat_cols)
    concat_dim = max_det * base_dim
    results = []

    window_start = 0.0
    while window_start < duration_s:
        remaining = duration_s - window_start
        if remaining < 0.1:
            break

        window_end = window_start + window_size_s

        # Find detections in this window
        mask = (file_df["det_start_time"] >= window_start) & (file_df["det_start_time"] < window_end)
        window_dets = file_df[mask]
        n_det = len(window_dets)

        if n_det > 0:
            # Sort by confidence (descending by default)
            window_dets = window_dets.sort_values(sort_by, ascending=sort_ascending)

            # Take top max_det detections
            was_truncated = n_det > max_det
            feats = window_dets[feat_cols].values[:max_det].astype(np.float32)
            k = len(feats)

            # L2-normalise each detection individually
            if normalize:
                norms = np.linalg.norm(feats, axis=1, keepdims=True)
                norms = np.maximum(norms, 1e-12)
                feats = feats / norms

            # Concatenate + zero-pad to fixed size
            concat = np.zeros(concat_dim, dtype=np.float32)
            concat[:k * base_dim] = feats.ravel()

            # L2-normalise the full concatenated vector
            if normalize:
                full_norm = np.linalg.norm(concat)
                if full_norm > 0:
                    concat = concat / full_norm

            results.append((window_start, concat, n_det, was_truncated))
        elif sentinel is not None:
            results.append((window_start, sentinel.copy(), 0, False))

        window_start += hop_size_s

    return results


print("embeddings_from_csv_concat() defined")
print(f"  MAX_DET={MAX_DET}, concat dim={EMBEDDING_DIM}")
print(f"  Sort: {SORT_BY} ({'asc' if SORT_ASCENDING else 'desc'})")
print(f"  Empty windows: sentinel = [{SENTINEL_VALUE}] * {EMBEDDING_DIM}")

## 4. Load and Inspect Stage 1 CSV

In [ ]:
# Load detections CSV
df = pd.read_csv(DETECTIONS_CSV)
print(f"Loaded {len(df)} detections from {DETECTIONS_CSV}")
print(f"Unique files: {df['source_id'].nunique()}")

# Load file index (needed for file durations)
file_index = pd.read_csv(FILE_INDEX_CSV)
print(f"Loaded {len(file_index)} files from {FILE_INDEX_CSV}")

# Build duration lookup
duration_lookup = dict(zip(file_index["source_id"], file_index["duration_s"]))
print(f"Duration lookup: {len(duration_lookup)} files")
print(f"Duration range: [{file_index['duration_s'].min():.3f}, {file_index['duration_s'].max():.3f}]s")

# Feature columns
FEAT_COLS = [f"feat_{i}" for i in range(BASE_DIM)]

# --- Detection count statistics ---
print(f"\n--- Detection Count per 0.5s Window ---")
det_grouped_tmp = df.groupby("source_id", sort=False)
all_counts = []
for _, row in file_index.iterrows():
    sid = row["source_id"]
    dur = row["duration_s"]
    if sid in det_grouped_tmp.groups:
        fdf = det_grouped_tmp.get_group(sid)
    else:
        fdf = pd.DataFrame(columns=df.columns)
    ws = 0.0
    while ws < dur:
        if dur - ws < 0.1:
            break
        mask = (fdf["det_start_time"] >= ws) & (fdf["det_start_time"] < ws + WINDOW_SIZE_S)
        all_counts.append(mask.sum())
        ws += HOP_SIZE_S

counts = np.array(all_counts)
nonempty = counts[counts > 0]
print(f"  Total windows: {len(counts)}")
print(f"  Non-empty: {len(nonempty)} ({100*len(nonempty)/len(counts):.1f}%)")
print(f"  Empty: {len(counts)-len(nonempty)} ({100*(len(counts)-len(nonempty))/len(counts):.1f}%)")
if len(nonempty) > 0:
    print(f"  Detections per non-empty window:")
    print(f"    mean={nonempty.mean():.1f}, median={np.median(nonempty):.0f}")
    for p in [75, 90, 95, 99]:
        print(f"    P{p}={np.percentile(nonempty, p):.0f}", end="  ")
    print(f"max={nonempty.max()}")
    n_trunc = (nonempty > MAX_DET).sum()
    print(f"  Windows exceeding MAX_DET={MAX_DET}: {n_trunc} ({100*n_trunc/len(nonempty):.1f}% of non-empty)")

# Test on one file
test_file = file_index["source_id"].iloc[0]
test_dur = duration_lookup[test_file]
test_df = df[df["source_id"] == test_file]
test_embs = embeddings_from_csv_concat(
    test_df, test_dur, FEAT_COLS,
    window_size_s=WINDOW_SIZE_S, hop_size_s=HOP_SIZE_S,
    max_det=MAX_DET, sort_by=SORT_BY, sort_ascending=SORT_ASCENDING,
    normalize=NORMALIZE_EMBEDDINGS, sentinel=SENTINEL_VECTOR,
)
n_with_det = sum(1 for _, _, n, _ in test_embs if n > 0)
n_sentinel = sum(1 for _, _, n, _ in test_embs if n == 0)
n_truncated = sum(1 for _, _, _, t in test_embs if t)
print(f"\nTest file: {test_file}")
print(f"  Duration: {test_dur:.3f}s")
print(f"  Detections in CSV: {len(test_df)}")
print(f"  Windows: {len(test_embs)} ({n_with_det} with dets, {n_sentinel} sentinel, {n_truncated} truncated)")
if test_embs:
    off, emb, nd, trunc = test_embs[0]
    tag = f"concat ({nd} dets{', TRUNCATED' if trunc else ''})" if nd > 0 else "SENTINEL"
    print(f"  First window: offset={off:.3f}s, n_det={nd}, dim={emb.shape[0]}, norm={np.linalg.norm(emb):.4f} ({tag})")

## 5. Initialize Hoplite Database

In [ ]:
# Handle existing database
db_path = Path(DB_PATH)

if db_path.exists():
    if DELETE_EXISTING_DB:
        print(f"Deleting existing database at {DB_PATH}...")
        shutil.rmtree(db_path)
    else:
        raise ValueError(
            f"Database exists at {DB_PATH}. "
            "Set DELETE_EXISTING_DB=True to overwrite."
        )

# USearch configuration
usearch_cfg = config_dict.ConfigDict({
    'embedding_dim': EMBEDDING_DIM,
    'dtype': USEARCH_DTYPE,
    'metric_name': USEARCH_METRIC,
    'expansion_add': 256,
    'expansion_search': 128,
})

print(f"Creating database with:")
print(f"  - Embedding dim: {usearch_cfg.embedding_dim}")
print(f"  - Dtype: {usearch_cfg.dtype}")
print(f"  - Metric: {usearch_cfg.metric_name}")

# Create database
db = sqlite_usearch_impl.SQLiteUsearchDB.create(
    db_path=str(db_path),
    usearch_cfg=usearch_cfg
)

print(f"\n\u2713 Database created at {DB_PATH}")

## 6. Store Metadata

In [ ]:
# Store model configuration metadata
model_config_dict = config_dict.ConfigDict({
    'model_key': 'batdetect2',
    'model_config': {
        'sample_rate': TARGET_SAMP_RATE,
        'embedding_dim': EMBEDDING_DIM,
        'base_dim': BASE_DIM,
        'feature_type': 'cnn_32dim_concat_max8',
        'extraction_method': 'stage1_csv_concat',
        'stage1_csv': DETECTIONS_CSV,
        'window_size_s': WINDOW_SIZE_S,
        'hop_size_s': HOP_SIZE_S,
        'max_detections_per_window': MAX_DET,
        'detection_sort_order': f"{SORT_BY}_{'asc' if SORT_ASCENDING else 'desc'}",
        'detection_threshold': DETECTION_THRESHOLD,
        'time_expansion': TIME_EXPANSION,
        'target_samp_rate': TARGET_SAMP_RATE,
        'chunk_size': CHUNK_SIZE,
        'normalize_embeddings': NORMALIZE_EMBEDDINGS,
        'sentinel_value': float(SENTINEL_VALUE),
    }
})
db.insert_metadata('model_config', model_config_dict)
print("Stored model_config metadata")

# Store audio source configuration
audio_source = source_info.AudioSourceConfig(
    dataset_name=DATASET_NAME,
    base_path=DATASET_BASE_PATH,
    file_glob=FILE_GLOB,
)
audio_sources = source_info.AudioSources((audio_source,))
audio_sources_dict = config_dict.ConfigDict(audio_sources.to_config_dict())
db.insert_metadata('audio_sources', audio_sources_dict)
print("Stored audio_sources metadata")

print("\n--- Metadata ---")
for k, v in model_config_dict.model_config.items():
    print(f"  {k}: {v}")

## 7. Batch Embedding Extraction

In [ ]:

# Batch insertion to hoplite


stats = {
    'files_processed': 0,
    'files_with_detections': 0,
    'total_windows': 0,
    'windows_with_detections': 0,
    'windows_sentinel': 0,
    'windows_truncated': 0,
    'total_detections_used': 0,
    'total_detections_available': 0,
    'errors': [],
}

# Group detections by source_id for efficient lookup
det_grouped = df.groupby("source_id", sort=False)

print(f"Processing {len(file_index)} files...")
print(f"Window: {WINDOW_SIZE_S}s, hop: {HOP_SIZE_S}s")
print(f"Concat: MAX_DET={MAX_DET}, dim={EMBEDDING_DIM}")
print(f"Sentinel for empty windows: [{SENTINEL_VALUE}] * {EMBEDDING_DIM}")

for _, row in tqdm(file_index.iterrows(), desc="Stage 2c (CSV->DB concat)", unit="file", total=len(file_index)):
    source_id = row["source_id"]
    duration_s = row["duration_s"]

    try:
        if source_id in det_grouped.groups:
            file_df = det_grouped.get_group(source_id)
        else:
            file_df = pd.DataFrame(columns=df.columns)

        windows = embeddings_from_csv_concat(
            file_df, duration_s, FEAT_COLS,
            window_size_s=WINDOW_SIZE_S, hop_size_s=HOP_SIZE_S,
            max_det=MAX_DET, sort_by=SORT_BY, sort_ascending=SORT_ASCENDING,
            normalize=NORMALIZE_EMBEDDINGS, sentinel=SENTINEL_VECTOR,
        )

        for offset_s, embedding, n_det, was_truncated in windows:
            offsets = np.asarray([offset_s], dtype=np.float16)
            source = db_interface.EmbeddingSource(
                dataset_name=DATASET_NAME,
                source_id=source_id,
                offsets=offsets,
            )
            db.insert_embedding(np.asarray(embedding, dtype=np.float32), source)

            stats['total_windows'] += 1
            if n_det > 0:
                stats['windows_with_detections'] += 1
                stats['total_detections_available'] += n_det
                stats['total_detections_used'] += min(n_det, MAX_DET)
                if was_truncated:
                    stats['windows_truncated'] += 1
            else:
                stats['windows_sentinel'] += 1

        stats['files_processed'] += 1
        if len(file_df) > 0:
            stats['files_with_detections'] += 1

    except Exception as e:
        stats['errors'].append((source_id, str(e)))
        if len(stats['errors']) <= 5:
            tqdm.write(f"[ERROR] {source_id}: {e}")

db.commit()
print(f"\n{'='*50}")
print("EMBEDDING COMPLETE")
print(f"{'='*50}")